In [1]:
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from typing import TypedDict, Literal, Annotated
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langgraph.checkpoint.memory import MemorySaver #for persistance memory (memory saver is for ram)

In [2]:
load_dotenv()
llm = ChatGroq(model = "openai/gpt-oss-120b")

In [9]:
from langgraph.graph.message import BaseMessage, add_messages
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages] #add_message is used because base message class supports this for reducer 

    # basemessage is the base class(parent class) for all message types in langgraph (aimessage,humanmessage,toolmessage)
    

In [4]:
def chat_node(state: ChatState):
    messages= state['messages']
    response = llm.invoke(messages)
    return {'messages': [response]}

In [6]:
checkpointer = MemorySaver() #object of memorysaver
graph = StateGraph(ChatState)

graph.add_node('chat_node',chat_node)

graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node', END)

chatbot = graph.compile(checkpointer=checkpointer)

In [7]:
from langchain_core.messages import HumanMessage

initial_state = {
    "messages": [HumanMessage(content="What is the capital of France?")]
}

chatbot.invoke(initial_state)

ValueError: Checkpointer requires one or more of the following 'configurable' keys: thread_id, checkpoint_ns, checkpoint_id

In [8]:
thread_id = '1' #tells who one is talking to llm currently

while True:
    user_message = input('Type here: ')
    print('User', user_message)
    if user_message.strip().lower() in ['exit', 'quit', 'bye']:
        break

    config = {'configurable': {'thread_id': thread_id}}
    response = chatbot.invoke({'messages': [HumanMessage(content=user_message)]}, config=config)

    print('AI', response['messages'][-1].content)

User capital of india
AI The capital of India is **New Delhi**.
User where it is located
AI New Delhi is situated in the northern part of India, within the National Capital Territory (NCT) of **Delhi**.  

- **Geographic coordinates:** roughly **28.6139° N latitude, 77.2090° E longitude**.  
- **Region:** It lies on the banks of the **Yamuna River**, about 200 km (≈125 mi) south of the Himalayan foothills and roughly 250 km (≈155 mi) west of the state of Uttar Pradesh’s capital, Lucknow.  
- **Surroundings:** The city is surrounded by the larger urban expanse of Delhi, which borders the Indian states of **Haryana** (to the west, north‑west, and south‑west) and **Uttar Pradesh** (to the east and north‑east).  
- **Accessibility:** Major highways (e.g., NH 48, NH 1) and the **Indira Gandhi International Airport** connect New Delhi to the rest of the country and the world.

In short, New Delhi is the central hub of the Delhi metropolitan area in northern India, serving as the nation’s pol